# Silver — ERP Customer Demographics
Birth date and gender per customer from the ERP.

`bronze.erp_cust_az12` → `silver.erp_customers`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.erp_cust_az12")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Clean customer id
Some ids carry a `NAS` prefix (`NASAW00011000`). Strip it so the id matches `crm_customers.customer_number` (`AW00011000`).

In [ ]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"), F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

### Invalidate future birth dates

In [ ]:
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None).otherwise(col("bdate"))
)

### Normalize gender
The ERP mixes codes and words (`F`, `FEMALE`, `M`, `MALE`).

In [ ]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customers")

In [ ]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10;